# Phase 1: NLP Tokenization & Text Preprocessing from Scratch

Welcome to the first notebook of your NLP learning journey! The objective here is to build fundamental text processing algorithms from scratch using standard Python. By building these algorithms yourself, you will gain a deep appreciation for the edge cases that standard libraries solve for you.

## Notebook Outline:
1. **Whitespace Tokenization** (and its limitations)
2. **Regex-Based Word Tokenization** (handling punctuation and contractions)
3. **Simple Sentence Tokenization** (detecting boundaries)
4. **Basic Suffix Stemming** (Porter-like suffix stripping)
5. **Byte Pair Encoding (BPE)** (building a vocabulary and encoding text)

---  
## 1. Whitespace Tokenization

The simplest way to tokenize text is to split on whitespace characters (spaces, tabs, newlines). Let's implement this first and see why it is insufficient for downstream NLP tasks.

In [ ]:
def tokenize_whitespace(text):
    """
    Splits text by whitespace characters.
    """
    # TODO: Implement whitespace splitting
    pass

In [ ]:
# Test cases to demonstrate limitations
sample_text = "NLP is state-of-the-art! However, it doesn't always work perfectly."
tokens = tokenize_whitespace(sample_text)
print("Tokens:", tokens)

# Note how punctuation remains attached to words (e.g. "state-of-the-art!", "perfectly.").
# Note how contractions are kept as one token ("doesn't").

---  
## 2. Regex-Based Word Tokenization

To handle punctuation better, we can use Python's built-in `re` (regular expressions) library to define what constitutes a token.

In [ ]:
import re

def tokenize_regex(text):
    """
    Tokenizes text using regular expressions.
    Should separate punctuation from words, while keeping contractions (like "don't") intact if possible.
    """
    # TODO: Define a pattern and extract tokens
    pattern = r"\w+(?:'\w+)?|[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~]"
    return re.findall(pattern, text)

In [ ]:
print("Regex Tokens:", tokenize_regex(sample_text))
# Output should ideally separate the exclamation mark and periods as their own tokens.

---  
## 3. Simple Sentence Tokenization

Splitting text into sentences is not as simple as splitting on periods. Think of titles/abbreviations like `Dr.`, `Mr.`, `e.g.`, or decimal points like `3.14`.

In [ ]:
def split_sentences(text):
    """
    Splits text into sentences, handling common sentence-ending punctuation (.?!)
    while attempting to avoid splitting on common abbreviations like Dr., Mr., etc.
    """
    # TODO: Implement sentence boundary detection
    pass

In [ ]:
sentence_sample = "Dr. Smith bought a laptop. It cost $1,000.50. What a deal!"
print("Sentences:", split_sentences(sentence_sample))

---  
## 4. Basic Suffix Stemming

Stemming is the process of reducing a word to its word stem or root. Let's implement a toy version of a suffix stemmer by stripping basic suffixes like `-ing`, `-ed`, `-ly`, and `-s`.

In [ ]:
def simple_stemmer(word):
    """
    Reduces a word to its root by stripping common suffixes like -ing, -ed, -ly, -s.
    """
    # TODO: Implement basic suffix stripping rules
    pass

In [ ]:
test_words = ["connections", "connecting", "connected", "happily", "dogs"]
for w in test_words:
    print(f"{w} -> {simple_stemmer(w)}")

---  
## 5. Byte Pair Encoding (BPE) from Scratch

BPE is a subword tokenization algorithm used by modern LLMs (like GPT). It starts with a character-level vocabulary and iteratively merges the most frequent pairs of adjacent characters/subwords.

In [ ]:
from collections import defaultdict

def get_vocab_counts(corpus):
    """
    Converts corpus of sentences into a dictionary of word frequencies, 
    representing words as tuples of characters with an end-of-word marker '</w>'.
    """
    vocab = defaultdict(int)
    for sentence in corpus:
        words = sentence.split()
        for word in words:
            char_tuple = tuple(list(word)) + ('</w>',)
            vocab[char_tuple] += 1
    return vocab

def get_stats(vocab):
    """
    Computes frequencies of all adjacent pairs of symbols in the vocabulary.
    """
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word)-1):
            pairs[(word[i], word[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab_in):
    """
    Merges all occurrences of the specified pair in the vocabulary.
    """
    vocab_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in vocab_in:
        word_str = ' '.join(word)
        # Perform the merge
        word_merged = p.sub(''.join(pair), word_str)
        vocab_out[tuple(word_merged.split())] = vocab_in[word]
    return vocab_out

In [ ]:
# TODO: Write a loop that executes BPE merging for 20 iterations
corpus = [
    "low low low low low lower lower newer newer newer newer newer newer"
]
vocab = get_vocab_counts(corpus)
num_merges = 10

print("Initial Vocab:", vocab)
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge #{i+1}: {best} -> Vocab state: {vocab}")